In [1]:
# Project: Technical Support Assistant (Agentic RAG System) using Autogen

# The Technical Support Assistant is an AI-powered, multi-agent system
# designed to provide accurate and context-aware answers to technical
# queries by combining Retrieval-Augmented Generation (RAG),
# web validation, and conversational memory.

# Key Features
# --Multi-Agent Architecture
# --Document-Based Question Answering (RAG)
# --Real-Time Web Validation
# --Conversational Memory
# --Using Tools


# How It Works
# --User submits a technical query
# --Planner Agent decomposes the task
# --RAG Agent retrieves relevant document context
# --Web Agent gathers supporting or validating information from internet
# --Answering Agent generates the final response using all inputs + memory

# Use Cases
# --IT Helpdesk automation
# --Customer support for technical products
# --Enterprise knowledge base querying

# Flow
# UserProxyAgent (User query)
#      ↓
# Planner Agent
#      ↓
# RAG Agent
#      ↓
# Web Agent
#      ↓
# Answering Agent (final reply to User query)


In [2]:
# 1. Install Dependencies
!pip install -q \
autogen \
openai \
chromadb \
pypdf \
ddgs \
tiktoken \
numpy \
pydantic \
nest_asyncio

In [3]:
#Setup OpenAI Key
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [4]:
#Build RAG (PDF → ChromaDB)
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings

# Read all the PDF documents in directory
pdf_directory = "/content/sample_data/"  # upload PDFs here

raw_text = ""

for file in os.listdir(pdf_directory):
    if file.endswith(".pdf"):
        reader = PdfReader(os.path.join(pdf_directory, file))
        for page in reader.pages:
            text = page.extract_text()
            if text:
                raw_text += text + "\n"

if not raw_text:
    raise ValueError("No PDFs found!")

# Define embedding function
openai_embedding_function = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small"
)


# Create chunks
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)

# Create chroma db client
chroma_client = chromadb.Client(
    Settings(anonymized_telemetry=False)
)

# Create collection WITH embeddings
collection = chroma_client.get_or_create_collection(
    name="tech_support",
    embedding_function=openai_embedding_function
)

# Add chunks to vector db
collection.add(
    documents=chunks,
    ids=[f"id_{i}" for i in range(len(chunks))]
)

print("RAG setup complete")


RAG setup complete


In [5]:
# Search Tools (RAG + Web)
from ddgs import DDGS

# Define rag_search() to search vector db
def rag_search(query: str) -> str:
    results = collection.query(query_texts=[query], n_results=3)
    return " ".join(results["documents"][0])

# Define web_search() to query search internet
def web_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=3):
            results.append(r["body"])
    return " ".join(results)


In [6]:
from autogen import AssistantAgent, UserProxyAgent, register_function


# to supress warning in google collab for google-generativeai.
#It's unnecessery warning thrown by collab.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# config_list: a list of dictionaries, each defining an LLM model, API key, etc
config_list = [
    {
        "model": "gpt-4o-mini",  # fast + cheap
        "api_key": os.environ["OPENAI_API_KEY"]
    }
]

# llm_config : A dictionary or autogen.LLMConfig object
# that specifies the Large Language Model (LLM) settings for the agent.
llm_config = {
    "config_list": config_list,
    "temperature": 0
}

In [7]:
# Define Autogen Agents

# 1. Planner Agent
planner = AssistantAgent(
    name="Planner",
    system_message="Break the user query into logical steps.",
    llm_config=llm_config
)

# 2. RAG Agent (Tool-enabled)
rag_agent = AssistantAgent(
    name="RAG_Agent",
    #system_message="Use retrieved document context to answer questions.",
    system_message=(
        "You are a helpful Technical Support assistant that retrieves relevant answers from product documents.\n" \
        "Please give summarised responses to user queries based on the technical context retrieved.\n" \
    ),
    llm_config=llm_config
)

rag_agent.register_function(
    function_map={"rag_search": rag_search}
)

# 3. Web Agent
web_agent = AssistantAgent(
    name="Web_Agent",
    system_message="Validate answers using internet search.",
    llm_config=llm_config
)

web_agent.register_function(
    function_map={"web_search": web_search}
)

# 4. Final answering_assistant who responds to user query
answering_agent = AssistantAgent(
    name="Final_Assistant",
    system_message="""
    Combine RAG results and web validation.
    Provide accurate, concise technical answers.
    """,
    llm_config=llm_config
)

In [8]:
# Define User Proxy Agent which receives the user query
# and initiates task executions to prepare the answer
user_proxy = UserProxyAgent(
    name="User",
    llm_config=False,
    #human_input_mode="ALWAYS",
    human_input_mode="NEVER",  # auto-execution
    max_consecutive_auto_reply=5,
    code_execution_config=False
)

In [9]:
# Add Global memory
memory = []

def save_memory(role, content):
    memory.append(f"{role}: {content}")

def get_memory(k=10):
    return "\n".join(memory[-k:])

In [10]:
# Orchestrate Multi-Agent Flow
from IPython.display import Markdown
def run_system(query):
    print("\n USER:", query)
    save_memory("User", query)

    # Inject memory
    past_context = get_memory()

    # Step 1: Plan
    plan_prompt = f"""
    Conversation history:
    {past_context}

    Break this query into steps:
    {query}
    """

    plan = planner.generate_reply(
        messages=[{"role": "user", "content": plan_prompt}]
    )
    print("\n PLAN:\n", plan)

    # Step 2: RAG search
    rag_context = rag_search(query)
    print("\n RAG CONTEXT:\n", rag_context[:500])

    # Step 3: Web search
    web_context = web_search(query)
    print("\n WEB CONTEXT:\n", web_context[:500])

    # Step 4: Final Answer (WITH MEMORY)
    final_prompt = f"""
    Conversation history:
    {past_context}

    User Question:
    {query}

    RAG Context:
    {rag_context}

    Web Context:
    {web_context}

    Provide a helpful and context-aware answer.
    """

    final_answer = answering_agent.generate_reply(
        messages=[{"role": "user", "content": final_prompt}]
    )

    # save final answer in merory so that we can retrieve later.
    save_memory("Assistant", final_answer)

    print("\n ***************************************")
    print(" FINAL ANSWER:\n")
    display(Markdown(final_answer))
    print("\n ***************************************\n")


In [11]:
from numpy import e
# Run Loop

while True:
    q = input("\nAsk your technical question (or 'exit'): ")

    if q.lower() == "exit":
        break

    run_system(q)


Ask your technical question (or 'exit'): In my windows 11 laptop, printer connected but not printing.

 USER: In my windows 11 laptop, printer connected but not printing.

 PLAN:
 1. Identify the device: Windows 11 laptop.
2. Confirm the connection status: Printer is connected.
3. Determine the issue: Printer is not printing.

 RAG CONTEXT:
 
 
 
 
4. Printer Not Working 
Problem 
The printer is connected but does not print. 
Troubleshooting Steps 
1. Ensure printer is powered on. 
2. Check USB or Wi-Fi connection. 
3. Set printer as default: 
o Settings → Bluetooth & Devices → Printers & Scanners 
4. Clear stuck print queue. 
5. Restart Print Spooler service. 
6. Reinstall or update printer drivers. 
7. Print a test page. 
 
5. Windows Update Failed 
Problem 
Windows updates are failing or stuck. 
Troubleshooting Steps 
1. Restart 

 WEB CONTEXT:
 This article covers common printer problems in Windows, including printer not found, print jobs stuck in the queue, printer spooler crashe

If your printer is connected to your Windows 11 laptop but not printing, follow these troubleshooting steps to resolve the issue:

1. **Check Printer Power**: Ensure that the printer is powered on and ready to print.

2. **Verify Connection**: 
   - If using a USB connection, check that the cable is securely connected to both the printer and the laptop.
   - If using Wi-Fi, ensure that the printer is connected to the same network as your laptop.

3. **Set as Default Printer**: 
   - Go to **Settings** → **Bluetooth & Devices** → **Printers & Scanners**.
   - Find your printer in the list and click on it, then select **Manage** and choose **Set as default**.

4. **Clear Print Queue**: 
   - Open the **Printers & Scanners** settings, select your printer, and click on **Open queue**. 
   - Cancel any stuck print jobs.

5. **Restart Print Spooler Service**: 
   - Press `Windows + R`, type `services.msc`, and hit Enter.
   - Find **Print Spooler**, right-click it, and select **Restart**.

6. **Update or Reinstall Printer Drivers**: 
   - Go to the printer manufacturer's website to download the latest drivers for your printer model.
   - Alternatively, you can uninstall the printer from **Printers & Scanners** and then reinstall it.

7. **Print a Test Page**: 
   - After performing the above steps, try printing a test page to see if the issue is resolved.

If the problem persists after trying these steps, consider checking for any software conflicts or consulting the printer's manual for additional troubleshooting specific to your model.


 ***************************************


Ask your technical question (or 'exit'): exit
